---

## **DIPLOME UNIVERSITAIRE SDA**


## **Projet Generative AI — Système Agentique d'Évaluation et d'Anticipation des Risques Climatiques**





Promotion 007

Avril 2026




**Corpus** : 10 rapports scientifiques (GIEC AR6, Copernicus, EM-DAT, NOAA, JRC, WMO, EU CELEX)

**Repo** : https://github.com/diegomerchanm/catastrophes-climatiques-rag

---

**Ce notebook est conçu pour être :**
- **reproductible** (chemins relatifs, seeds fixées)
- **idempotent** (relançable sans recalculer si les fichiers existent déjà)
- **traçable** (quality gates go/no-go explicites)

**Convention :** chaque cellule de code doit produire une sortie visible. Si aucune sortie naturelle, ajouter un `print()` de vérification.

---

# NOTEBOOK 6 — Comparatifs & MLflow

**Auteur :** Xia Bizot

---

### Objectif

Benchmark complet des configurations du système RAG : pondérations BM25/Dense, taille des chunks, température, reranking, Map-Reduce. Chaque configuration est trackée dans MLflow.

---

### Plan du notebook

| Section | Contenu |
|---|---|
| 1. Configuration | Imports, chemins, seed, constantes, MLflow |
| 2. Set de questions de test | 10 questions identiques pour tous les configs |
| 3. Comparatif retrievers | Dense vs BM25 vs Hybride vs Reranking |
| 4. Comparatif chunks | 500 vs 800 vs 1500 vs 2000 |
| 5. Comparatif températures | 0 vs 0.2 vs 0.5 vs 0.7 |
| 6. A/B testing prompts | v1.0 vs v2.0 |
| 7. Résultats | Tableaux, visualisations, MLflow |
| 8. Conclusions | Quality gate, recommandations |

---

### Hypothèse testable

> La configuration hybride BM25+Dense avec reranking offre le meilleur compromis pertinence/coût par rapport aux autres configurations testées.

---

---

## 1. Configuration

In [ ]:
import os
import sys
import time
import logging
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

SEED = 42
np.random.seed(SEED)
notebook_start_time = time.time()

BASE = Path('..')
OUTPUT_DIR = BASE / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(BASE))

from dotenv import load_dotenv
load_dotenv(BASE / '.env')

import mlflow
mlflow.set_experiment('nb6-comparatifs-rag')

from src.config import (
    CHUNK_SIZE, CHUNK_OVERLAP, RETRIEVER_K, RETRIEVER_FETCH_K,
    BM25_WEIGHT, DENSE_WEIGHT, AGENT_CONFIGS, MODEL_PRICING,
)
from src.agents.agent import run_agent, get_token_summary, reset_token_counter

print(f'CHUNK_SIZE    : {CHUNK_SIZE}')
print(f'RETRIEVER_K   : {RETRIEVER_K}')
print(f'BM25/Dense    : {BM25_WEIGHT}/{DENSE_WEIGHT}')
print(f'MLflow        : {mlflow.__version__}')
print('>> 1. Configuration : OK')

---

## 2. Set de questions de test

Mêmes questions pour toutes les configurations — permet une comparaison juste.

In [ ]:
QUESTIONS_TEST = [
    'Quelles régions sont les plus vulnérables aux inondations selon le GIEC ?',
    'Que dit le rapport CELEX sur les inondations ?',
    'Quelles sont les recommandations du GIEC pour l\'adaptation ?',
    'Compare les événements climatiques de 2022 et 2023',
    'Quel est l\'impact des feux de forêt en Europe en 2024 ?',
    'Quelles catastrophes ont touché la Méditerranée en 2023 ?',
    'Que dit Copernicus sur le climat européen en 2023 ?',
    'Quel est le risque d\'inondation à Marseille ?',
    'Quelles sont les tendances des précipitations extrêmes ?',
    'Que dit le rapport EM-DAT sur les catastrophes naturelles en 2023 ?',
]

print(f'{len(QUESTIONS_TEST)} questions de test définies')
for i, q in enumerate(QUESTIONS_TEST):
    print(f'  Q{i+1}: {q[:60]}')
print('>> 2. Questions de test : OK')

---

## 3. Comparatif des pondérations BM25 / Dense

Tester 3 configurations : BM25 fort (70/30), équilibré (50/50), Dense fort (30/70).

In [ ]:
# Note : ce test nécessite de modifier BM25_WEIGHT/DENSE_WEIGHT dynamiquement
# Pour l'instant on teste avec la config par défaut (50/50)
# Les autres configs seront testées en modifiant config.py

configs_poids = [
    ('BM25=30/Dense=70', 0.3, 0.7),
    ('BM25=50/Dense=50', 0.5, 0.5),
    ('BM25=70/Dense=30', 0.7, 0.3),
]

results_poids = []
question_test = QUESTIONS_TEST[1]  # 'Que dit le rapport CELEX sur les inondations ?'

for nom_config, bm25_w, dense_w in configs_poids:
    reset_token_counter()
    t0 = time.time()
    reponse = run_agent(question_test)
    duree = round(time.time() - t0, 2)
    tokens = get_token_summary()
    
    results_poids.append({
        'config': nom_config,
        'tokens': tokens['total_tokens'],
        'cout_usd': tokens['estimated_cost_usd'],
        'duree_s': duree,
        'longueur_reponse': len(reponse),
    })
    
    with mlflow.start_run(run_name=f'poids_{nom_config}'):
        mlflow.log_param('config', nom_config)
        mlflow.log_param('bm25_weight', bm25_w)
        mlflow.log_param('dense_weight', dense_w)
        mlflow.log_metric('tokens', tokens['total_tokens'])
        mlflow.log_metric('cout_usd', tokens['estimated_cost_usd'])
        mlflow.log_metric('duree_s', duree)
    
    print(f'  {nom_config:25s} → {tokens["total_tokens"]:6d} tokens, {duree}s')

df_poids = pd.DataFrame(results_poids)
print(f'\n{df_poids.to_string()}')
print('>> 3. Comparatif pondérations : OK')

### Analyse

**Ce qu'on observe :**
- [Quelle pondération donne les meilleurs résultats sur la question CELEX (mot-clé exact) ?]
- [Le BM25 fort retrouve-t-il mieux le document CELEX ?]

---

---

## 4. Comparatif des températures

Impact de la température sur la qualité et la créativité des réponses.

In [ ]:
configs_temp = [0, 0.2, 0.5, 0.7]
question_temp = QUESTIONS_TEST[0]  # 'Quelles régions sont les plus vulnérables...'

results_temp = []

for temp in configs_temp:
    reset_token_counter()
    t0 = time.time()
    reponse = run_agent(question_temp)
    duree = round(time.time() - t0, 2)
    tokens = get_token_summary()
    
    results_temp.append({
        'temperature': temp,
        'tokens': tokens['total_tokens'],
        'cout_usd': tokens['estimated_cost_usd'],
        'duree_s': duree,
        'longueur_reponse': len(reponse),
    })
    
    with mlflow.start_run(run_name=f'temp_{temp}'):
        mlflow.log_param('temperature', temp)
        mlflow.log_metric('tokens', tokens['total_tokens'])
        mlflow.log_metric('cout_usd', tokens['estimated_cost_usd'])
        mlflow.log_metric('duree_s', duree)
        mlflow.log_metric('longueur_reponse', len(reponse))
    
    print(f'  temp={temp} → {tokens["total_tokens"]:6d} tokens, {len(reponse)} car, {duree}s')

df_temp = pd.DataFrame(results_temp)
print(f'\n{df_temp.to_string()}')
print('>> 4. Comparatif températures : OK')

### Analyse

**Ce qu'on observe :**
- [La température impacte-t-elle la longueur de la réponse ?]
- [Les réponses à temperature=0 sont-elles plus factuelles ?]

---

---

## 5. A/B testing des prompts

Comparer le prompt v1.0 (instructions générales) vs v2.0 (protocole d'analyse imposé).

In [ ]:
from src.prompts.agent_prompts import get_prompt, list_versions

question_ab = QUESTIONS_TEST[5]  # 'Quelles catastrophes ont touché la Méditerranée en 2023 ?'
results_ab = []

for version in list_versions():
    prompt = get_prompt(version)
    reset_token_counter()
    t0 = time.time()
    reponse = run_agent(question_ab)
    duree = round(time.time() - t0, 2)
    tokens = get_token_summary()
    
    results_ab.append({
        'prompt_version': version,
        'tokens': tokens['total_tokens'],
        'cout_usd': tokens['estimated_cost_usd'],
        'duree_s': duree,
        'longueur_reponse': len(reponse),
        'prompt_length': len(prompt),
    })
    
    with mlflow.start_run(run_name=f'prompt_{version}'):
        mlflow.log_param('prompt_version', version)
        mlflow.log_param('prompt_length', len(prompt))
        mlflow.log_metric('tokens', tokens['total_tokens'])
        mlflow.log_metric('cout_usd', tokens['estimated_cost_usd'])
        mlflow.log_metric('duree_s', duree)
    
    print(f'  {version} → {tokens["total_tokens"]:6d} tokens, {len(reponse)} car, {duree}s')

df_ab = pd.DataFrame(results_ab)
print(f'\n{df_ab.to_string()}')
print('>> 5. A/B testing prompts : OK')

### Analyse

**Ce qu'on observe :**
- [Le prompt v2.0 (protocole imposé) génère-t-il des réponses plus structurées ?]
- [Le coût est-il significativement différent ?]

---

---

## 6. Résultats

### 6.1. Tableau synthétique global

In [ ]:
# Sauvegarde
df_poids.to_csv(OUTPUT_DIR / 'NB6_comparatif_poids.csv', index=False)
df_temp.to_csv(OUTPUT_DIR / 'NB6_comparatif_temperature.csv', index=False)
df_ab.to_csv(OUTPUT_DIR / 'NB6_comparatif_prompts.csv', index=False)

print('  [OK] NB6_comparatif_poids.csv')
print('  [OK] NB6_comparatif_temperature.csv')
print('  [OK] NB6_comparatif_prompts.csv')

In [ ]:
# Visualisation comparative
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Pondérations
df_poids.plot(x='config', y='tokens', kind='bar', ax=axes[0], color='steelblue', legend=False)
axes[0].set_title('Tokens par pondération BM25/Dense')
axes[0].set_xlabel('Configuration')
axes[0].tick_params(axis='x', rotation=45)

# Températures
df_temp.plot(x='temperature', y='longueur_reponse', kind='bar', ax=axes[1], color='#e74c3c', legend=False)
axes[1].set_title('Longueur réponse par température')
axes[1].set_xlabel('Temperature')

# Prompts
df_ab.plot(x='prompt_version', y='tokens', kind='bar', ax=axes[2], color='#2ecc71', legend=False)
axes[2].set_title('Tokens par version de prompt')
axes[2].set_xlabel('Version')

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'NB6_comparatifs.png', dpi=200, bbox_inches='tight')
print('  [OK] NB6_comparatifs.png')
plt.show()

### 6.2. Dashboard MLflow

In [ ]:
try:
    runs = mlflow.search_runs(experiment_names=['nb6-comparatifs-rag'], max_results=20)
    if len(runs) > 0:
        cols = [c for c in runs.columns if c.startswith('params.') or c.startswith('metrics.')]
        print(runs[cols[:8]].to_string())
    else:
        print('Aucun run trouvé.')
except Exception as e:
    print(f'Erreur MLflow : {e}')

print('\nDashboard : mlflow ui --host 127.0.0.1 --port 8080')
print('>> 6. Résultats : OK')

---

## 7. Conclusions

### Quality gate finale

| Constat | Preuve | Décision |
|---|---|---|
| [Pondération optimale] | Section 3 | [Garder 50/50 ou ajuster] |
| [Impact température] | Section 4 | [Garder 0.2 ou ajuster] |
| [Prompt v1 vs v2] | Section 5 | [Garder v1 ou passer v2] |

---

### Hypothèse : [validée / invalidée]

[À compléter après exécution.]

---

### Limites

- Les tests de pondération nécessitent de modifier config.py entre les runs
- La température est celle de l'orchestrateur, pas du retriever
- Le A/B testing utilise le même agent (pas de switch dynamique de prompt)

---

### Axes d'amélioration

- Benchmark sur les 10 questions complètes (pas une seule)
- Ajout du comparatif reranking vs sans reranking
- Ajout du Map-Reduce vs contexte direct
- Métriques de pertinence (LLM-as-judge ou humain)

In [ ]:
elapsed = round(time.time() - notebook_start_time, 2)
print(f'Temps total du notebook : {elapsed}s')
print('>> NOTEBOOK 6 TERMINÉ')